In [0]:
```python
# ============================================================
# MAP_ORDER_ITEM — STEP 1
# CONFIGURATION
# ============================================================

import uuid

# ============================================================
# PROJECT CONFIGURATION
# Replace placeholders with your environment-specific values.
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

# Raw Delta
RAW_PATH = "gs://YOUR_RAW_BUCKET_NAME/order_items/"

# BigQuery dataset
BQ_DATASET = "Gcp_Curated"

# Target and staging tables
TARGET_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.MAP_ORDER_ITEM"
STAGE_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.MAP_ORDER_ITEM_STAGE"

# Existing warehouse tables
FACT_ORDER_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.FACT_ORDER"
DIM_PRODUCT_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.DIM_PRODUCT"

# Temporary bucket for the BigQuery connector
BQ_TEMP_BUCKET = "gs://YOUR_BQ_TEMP_BUCKET/"

# Watermark table
WATERMARK_TABLE = "YOUR_CATALOG.YOUR_WATERMARK_TABLE"

# Pipeline metadata
PIPELINE_NAME = "map_order_item_pipeline"
SOURCE_SYSTEM = "CloudSQL"

# Generate a unique pipeline run ID
RUN_ID = str(uuid.uuid4())

print("MAP_ORDER_ITEM pipeline initialized")
print("RUN_ID:", RUN_ID)
```

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 2
# READ RAW DELTA
# ============================================================

raw_order_items = (
    spark.read
    .format("delta")
    .load(RAW_PATH)
)

print("Total raw order-item records:", raw_order_items.count())

raw_order_items.printSchema()

display(
    raw_order_items
    .orderBy("order_id")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 3
# READ LAST WATERMARK
# ============================================================
from pyspark.sql import functions as F
watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(F.col("table_name") == "MAP_ORDER_ITEM")
    .filter(F.col("status") == "SUCCESS")
)

lwv = (
    watermark_df
    .select("lwv")
    .first()["lwv"]
)

print("MAP_ORDER_ITEM LWV:", lwv)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 4
# INCREMENTAL EXTRACTION
# ============================================================

order_item_incremental = (
    raw_order_items
    .filter(F.col("updated_at") > F.lit(lwv))
)

incremental_count = order_item_incremental.count()

print("MAP_ORDER_ITEM incremental records:", incremental_count)

display(
    order_item_incremental
    .orderBy("order_item_id")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 5
# REMOVE EXACT DUPLICATES
# ============================================================

before_dedup = order_item_incremental.count()

order_item_dedup = (
    order_item_incremental
    .dropDuplicates()
)

after_dedup = order_item_dedup.count()

print("Records before exact dedup:", before_dedup)
print("Records after exact dedup :", after_dedup)
print("Exact duplicates removed  :", before_dedup - after_dedup)

In [0]:
# Check business-key duplicates

duplicate_order_items = (
    order_item_dedup
    .groupBy("order_item_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_order_items.count()

print("Duplicate order_item_id values:", duplicate_count)

if duplicate_count > 0:
    display(
        duplicate_order_items
        .orderBy(F.col("count").desc())
    )
else:
    print("No duplicate order_item_id values found.")

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 6
# READ SURROGATE KEY TABLES FROM BIGQUERY
# ============================================================

PROJECT_ID = "gcp-data-engineering-504911"

fact_order = (
    spark.read
    .format("bigquery")
    .option("project", PROJECT_ID)
    .option("parentProject", PROJECT_ID)
    .option(
        "table",
        f"{PROJECT_ID}.Gcp_Curated.FACT_ORDER"
    )
    .load()
    .select(
        "order_id",
        "order_sk"
    )
)

dim_product = (
    spark.read
    .format("bigquery")
    .option("project", PROJECT_ID)
    .option("parentProject", PROJECT_ID)
    .option(
        "table",
        f"{PROJECT_ID}.Gcp_Curated.DIM_PRODUCT"
    )
    .load()
    .select(
        "product_id",
        "product_sk"
    )
)

print("FACT_ORDER records:", fact_order.count())
print("DIM_PRODUCT records:", dim_product.count())

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 7
# ENRICH WITH SURROGATE KEYS
# ============================================================

map_order_item_enriched = (
    order_item_dedup.alias("oi")

    # Get order_sk
    .join(
        fact_order.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "left"
    )

    # Get product_sk
    .join(
        dim_product.alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "left"
    )

    .select(
        F.col("oi.order_item_id"),
        F.col("oi.order_id"),
        F.col("o.order_sk"),
        F.col("oi.product_id"),
        F.col("p.product_sk"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        F.col("oi.discount_amount"),
        F.col("oi.line_total"),
        F.col("oi.created_at"),
        F.col("oi.updated_at")
    )
)

print(
    "MAP_ORDER_ITEM enriched records:",
    map_order_item_enriched.count()
)

display(
    map_order_item_enriched
    .orderBy("order_item_id")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 9
# SEPARATE VALID AND UNRESOLVED RECORDS
# ============================================================

map_order_item_clean = (
    map_order_item_enriched
    .filter(
        F.col("order_sk").isNotNull() &
        F.col("product_sk").isNotNull()
    )
)

map_order_item_unresolved = (
    map_order_item_enriched
    .filter(
        F.col("order_sk").isNull() |
        F.col("product_sk").isNull()
    )
)

clean_count = map_order_item_clean.count()
unresolved_count = map_order_item_unresolved.count()

print("Clean MAP_ORDER_ITEM records:", clean_count)
print("Unresolved records:", unresolved_count)
print("Total:", clean_count + unresolved_count)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 10
# IDENTIFY UNRESOLVED RECORD REASON
# ============================================================

map_order_item_unresolved = (
    map_order_item_unresolved
    .withColumn(
        "reject_reason",
        F.when(
            F.col("order_sk").isNull() &
            F.col("product_sk").isNull(),
            F.lit("MISSING_ORDER_SK_AND_PRODUCT_SK")
        )
        .when(
            F.col("order_sk").isNull(),
            F.lit("MISSING_ORDER_SK")
        )
        .when(
            F.col("product_sk").isNull(),
            F.lit("MISSING_PRODUCT_SK")
        )
        .otherwise(
            F.lit("UNKNOWN")
        )
    )
)

display(
    map_order_item_unresolved
    .groupBy("reject_reason")
    .count()
    .orderBy("reject_reason")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 11
# CREATE ORDER ITEM SURROGATE KEY
# ============================================================

map_order_item_stage = (
    map_order_item_clean
    .withColumn(
        "order_item_sk",
        F.abs(
            F.xxhash64(
                F.col("order_item_id").cast("string")
            )
        )
    )
    .select(
        "order_item_sk",
        "order_item_id",
        "order_sk",
        "order_id",
        "product_sk",
        "product_id",
        "quantity",
        "unit_price",
        "discount_amount",
        "line_total",
        "created_at",
        "updated_at"
    )
)

print(
    "MAP_ORDER_ITEM stage records:",
    map_order_item_stage.count()
)

display(
    map_order_item_stage
    .orderBy("order_item_id")
)

In [0]:
# ============================================================
# VERIFY ORDER ITEM SURROGATE KEY
# ============================================================

map_order_item_stage.select(
    F.count("*").alias("total_records"),
    F.countDistinct("order_item_sk").alias("distinct_order_item_sk"),
    F.countDistinct("order_item_id").alias("distinct_order_item_id")
).show()

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 12
# CREATE BIGQUERY STAGE TABLE
# ============================================================
import requests
from google.auth.transport.requests import Request
PROJECT_ID = "gcp-data-engineering-504911"

query = f"""
CREATE OR REPLACE TABLE
`{PROJECT_ID}.Gcp_Curated.MAP_ORDER_ITEM_STAGE`
(
    order_item_sk BIGINT NOT NULL,
    order_item_id BIGINT NOT NULL,

    order_sk BIGINT NOT NULL,
    order_id BIGINT NOT NULL,

    product_sk BIGINT NOT NULL,
    product_id BIGINT NOT NULL,

    quantity INT64,
    unit_price NUMERIC(10,2),
    discount_amount NUMERIC(10,2),
    line_total NUMERIC(12,2),

    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
"""

provider = dbutils.credentials.getServiceCredentialsProvider("bigquery")
provider.refresh(Request())

headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

url = (
    f"https://bigquery.googleapis.com/"
    f"bigquery/v2/projects/{PROJECT_ID}/queries"
)

body = {
    "query": query,
    "useLegacySql": False,
    "location": "asia-south2"
}

response = requests.post(
    url,
    headers=headers,
    json=body
)

print("HTTP status:", response.status_code)
print(response.json())

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 13
# WRITE STAGE DATA TO BIGQUERY
# ============================================================

(
    map_order_item_stage
    .write
    .format("bigquery")
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .mode("overwrite")
    .save(STAGE_TABLE)
)

print("MAP_ORDER_ITEM stage data written successfully.")

In [0]:
# ============================================================
# VERIFY STAGE COUNT
# ============================================================

stage_check = (
    spark.read
    .format("bigquery")
    .option("project", PROJECT_ID)
    .option("parentProject", PROJECT_ID)
    .option("table", STAGE_TABLE)
    .load()
)

print("BigQuery stage records:", stage_check.count())

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 15
# MERGE STAGE INTO TARGET
# ============================================================

PROJECT_ID = "gcp-data-engineering-504911"

TARGET_TABLE = f"{PROJECT_ID}.Gcp_Curated.MAP_ORDER_ITEM"
STAGE_TABLE = f"{PROJECT_ID}.Gcp_Curated.MAP_ORDER_ITEM_STAGE"

query = f"""
MERGE `{TARGET_TABLE}` T
USING `{STAGE_TABLE}` S
ON T.order_item_id = S.order_item_id

WHEN MATCHED THEN
    UPDATE SET
        order_sk = S.order_sk,
        order_id = S.order_id,
        product_sk = S.product_sk,
        product_id = S.product_id,
        quantity = S.quantity,
        unit_price = S.unit_price,
        discount_amount = S.discount_amount,
        line_total = S.line_total,
        created_at = S.created_at,
        updated_at = S.updated_at,
        dw_updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
    INSERT (
        order_item_sk,
        order_item_id,
        order_sk,
        order_id,
        product_sk,
        product_id,
        quantity,
        unit_price,
        discount_amount,
        line_total,
        created_at,
        updated_at,
        dw_created_at,
        dw_updated_at
    )
    VALUES (
        S.order_item_sk,
        S.order_item_id,
        S.order_sk,
        S.order_id,
        S.product_sk,
        S.product_id,
        S.quantity,
        S.unit_price,
        S.discount_amount,
        S.line_total,
        S.created_at,
        S.updated_at,
        CURRENT_TIMESTAMP(),
        CURRENT_TIMESTAMP()
    )
"""

provider = dbutils.credentials.getServiceCredentialsProvider("bigquery")
provider.refresh(Request())

headers = {
    "Authorization": f"Bearer {provider.token}",
    "X-Goog-User-Project": PROJECT_ID,
    "Content-Type": "application/json"
}

url = (
    f"https://bigquery.googleapis.com/"
    f"bigquery/v2/projects/{PROJECT_ID}/queries"
)

body = {
    "query": query,
    "useLegacySql": False,
    "location": "asia-south2"
}

response = requests.post(
    url,
    headers=headers,
    json=body
)

print("HTTP status:", response.status_code)
print(response.json())

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 16
# QUARANTINE UNRESOLVED RECORDS
# ============================================================

REJECT_TABLE = (
    f"{PROJECT_ID}.Gcp_Curated.DQ_REJECTED_RECORDS"
)

map_order_item_quarantine = (
    map_order_item_unresolved

    .withColumn(
        "dq_rule_id",
        F.when(
            F.col("reject_reason") == "MISSING_ORDER_SK",
            F.lit("MAP_001")
        )
        .when(
            F.col("reject_reason") == "MISSING_PRODUCT_SK",
            F.lit("MAP_002")
        )
        .when(
            F.col("reject_reason") == "MISSING_ORDER_SK_AND_PRODUCT_SK",
            F.lit("MAP_003")
        )
    )

    .withColumn(
        "dq_rule_name",
        F.when(
            F.col("reject_reason") == "MISSING_ORDER_SK",
            F.lit("Order Surrogate Key Not Found")
        )
        .when(
            F.col("reject_reason") == "MISSING_PRODUCT_SK",
            F.lit("Product Surrogate Key Not Found")
        )
        .when(
            F.col("reject_reason") == "MISSING_ORDER_SK_AND_PRODUCT_SK",
            F.lit("Order and Product Surrogate Keys Not Found")
        )
    )

    .withColumn(
        "record_key",
        F.col("order_item_id").cast("string")
    )

    .withColumn(
        "record_data",
        F.to_json(
            F.struct(
                "order_item_id",
                "order_id",
                "product_id",
                "quantity",
                "unit_price",
                "discount_amount",
                "line_total",
                "created_at",
                "updated_at"
            )
        )
    )

    .withColumn(
        "failure_reason",
        F.col("reject_reason")
    )

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "table_name",
        F.lit("MAP_ORDER_ITEM")
    )

    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )

    .withColumn(
        "pipeline_name",
        F.lit(PIPELINE_NAME)
    )

    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )

    .withColumn(
        "reject_id",
        F.monotonically_increasing_id()
    )

    .select(
        "reject_id",
        "run_id",
        "table_name",
        "dq_rule_id",
        "dq_rule_name",
        "record_key",
        "record_data",
        "failure_reason",
        "source_system",
        "pipeline_name",
        "rejected_at"
    )
)

In [0]:
print(
    "Records to quarantine:",
    map_order_item_quarantine.count()
)

display(
    map_order_item_quarantine
    .groupBy(
        "dq_rule_id",
        "dq_rule_name"
    )
    .count()
    .orderBy("dq_rule_id")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 17
# WRITE QUARANTINE RECORDS
# ============================================================

(
    map_order_item_quarantine
    .write
    .format("bigquery")
    .option("temporaryGcsBucket", BQ_TEMP_BUCKET)
    .mode("append")
    .save(REJECT_TABLE)
)

print("MAP_ORDER_ITEM quarantine records written successfully.")

In [0]:
# ============================================================
# VERIFY QUARANTINE
# ============================================================

quarantine_check = (
    spark.read
    .format("bigquery")
    .option("project", PROJECT_ID)
    .option("parentProject", PROJECT_ID)
    .option("table", REJECT_TABLE)
    .load()
    .filter(
        (F.col("run_id") == RUN_ID) &
        (F.col("table_name") == "MAP_ORDER_ITEM")
    )
)

print(
    "MAP_ORDER_ITEM quarantined records:",
    quarantine_check.count()
)

display(
    quarantine_check
    
    .orderBy("dq_rule_id")
)

In [0]:
# ============================================================
# MAP_ORDER_ITEM — STEP 18
# UPDATE WATERMARK AFTER SUCCESSFUL PROCESSING
# ============================================================

new_lwv = (
    order_item_incremental
    .agg(
        F.max("updated_at").alias("max_updated_at")
    )
    .first()["max_updated_at"]
)

print("Previous LWV:", lwv)
print("New LWV:", new_lwv)

if new_lwv is not None:

    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET
            lwv = TIMESTAMP('{new_lwv}'),
            status = 'SUCCESS'
        WHERE table_name = 'MAP_ORDER_ITEM'
    """)

    print("MAP_ORDER_ITEM watermark updated successfully.")

else:

    print(
        "No incremental records found. "
        "MAP_ORDER_ITEM watermark was not changed."
    )

In [0]:
# ============================================================
# VERIFY UPDATED WATERMARK
# ============================================================

spark.sql(f"""
    SELECT
        table_name,
        status,
        lwv
    FROM {WATERMARK_TABLE}
    WHERE table_name = 'MAP_ORDER_ITEM'
""").show(truncate=False)